<a href="https://colab.research.google.com/github/baricoh1/Dashboard/blob/main/AI_Papers_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Libraries

In [ ]:
!pip install -q google-genai gradio httpx sentence-transformers scikit-learn

# Search Engine

In [ ]:
import httpx
import numpy as np
import gradio as ui
from google import genai
from google.genai import types
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize the Gemini Client (Still used for heavy full-text reading and generation)
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 2. Initialize the Local Embedding Model
# This downloads a ~90MB footprint model once and runs entirely on your local CPU/GPU memory.
print("🧠 Loading local sentence-transformer model ('all-MiniLM-L6-v2')...")
local_embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Hardcoded Knowledge Base with Pre-Compiled Abstracts
PDF_DB = [
    {
        "title": "Artificial Intelligence for Long-Term Robot Autonomy: A Survey",
        "url": "https://www.robots.ox.ac.uk/~mobile/Papers/RAL18_kunze.pdf",
        "abstract": "Autonomous systems will play an essential role in many applications across diverse domains including space, marine, air, field, road, and service robotics. They will assist us in our daily routines and perform dangerous, dirty, and dull tasks. However, enabling robotic systems to perform autonomously in complex, real-world scenarios over extended time periods (i.e., weeks, months, or years) poses many challenges. Some of these have been investigated by subdisciplines of Artificial Intelligence (AI) including navigation and mapping, perception, knowledge representation and reasoning, planning, interaction, and learning. The different subdisciplines have developed techniques that, when re-integrated within an autonomous system, can enable robots to operate effectively in complex, long-term scenarios. In this letter, we survey and discuss AI techniques as “enablers” for long-term robot autonomy, current progress in integrating these techniques within long-running robotic systems, and the future challenges and opportunities for AI in long-term autonomy."
    },
    {
        "title": "Indoor Air Quality Detection Robot Model Based on the Internet of Things (IoT)",
        "url": "https://arxiv.org/pdf/2505.19600",
        "abstract": "This paper presents the design, implementation, and evaluation of an IoT-based robotic system for mapping and monitoring indoor air quality. The primary objective was to develop a mobile robot capable of autonomously mapping a closed environment, detecting concentrations of CO2, volatile organic compounds (VOCs), smoke, temperature, and humidity, and transmitting real-time data to a web interface. The system integrates a set of sensors (SGP30, MQ-2, DHT11, VL53L0X, MPU6050) with an ESP32 microcontroller. It employs a mapping algorithm for spatial data acquisition and utilizes a Mamdani fuzzy logic system for air quality classification. Empirical tests in a model room demonstrated average localization errors below 5%, actuator motion errors under 2%, and sensor measurement errors within 12% across all modalities. The contributions of this work include: (1) a low-cost, integrated IoT robotic platform for simultaneous mapping and air quality detection; (2) a web-based user interface for real-time visualization and control; and (3) validation of system accuracy under laboratory conditions."
    },
    {
        "title": "A Review of Sensing Technologies for Indoor Autonomous Mobile Robots",
        "url": "https://mdpi-res.com/d_attachment/sensors/sensors-24-01222/article_deploy/sensors-24-01222-v2.pdf?version=1708311024",
        "abstract": "As a fundamental issue in robotics academia and industry, indoor autonomous mobile robots (AMRs) have been extensively studied. For AMRs, it is crucial to obtain information about their working environment and themselves, which can be realized through sensors and the extraction of corresponding information from the measurements of these sensors. The application of sensing technologies can enable mobile robots to perform localization, mapping, target or obstacle recognition, and motion tasks, etc. This paper reviews sensing technologies for autonomous mobile robots in indoor scenes. The benefits and potential problems of using a single sensor in application are analyzed and compared, and the basic principles and popular algorithms used in processing these sensor data are introduced. In addition, some mainstream technologies of multi-sensor fusion are introduced. Finally, this paper discusses the future development trends in the sensing technology for autonomous mobile robots in indoor scenes, as well as the challenges in the practical application environments."
    },
    {
        "title": "Design of a new method for occupancy monitoring in smart home care with autonomous mobile robot within Internet of Things",
        "url": "https://www.nature.com/articles/s41598-025-16806-8.pdf",
        "abstract": "The integration of autonomous mobile robots in Smart Home and their secure communication within Internet of Things with 5G networks represents a transformative shift towards more efficient, responsive, and adaptable healthcare and service delivery systems to support independent living for older people at home. This article presents a unique proposal for the possibility of implementing interoperability and secure data transmission within the communication between autonomous mobile robots and building automation technology in a Smart Home using 5G networks and also presents a novel design and application of a time-ahead concentration prediction method for sending presence and occupancy information in monitored Smart Home Care spaces without the use of cameras to an autonomous mobile robot for time-ahead detection of deviations from the daily routine. In this study, nonlinear input-output neural network models and nonlinear autoregressive neural network model with exogenous inputs neural network models with the following best results were used. Levenberg-Marquardt algorithm, Bayes regularization algorithm and Scaled Conjugate Gradient algorithm have been used as learning algorithms. Measured waveforms of operational and technical variables for indoor environmental quality (temperature, relative humidity, light intensity and concentration) and binary information from magnetic contacts placed on windows and doors (opening/closing of windows and doors) were used to monitor the presence of occupants in the Smart Home Care with autonomous mobile robots without the use of cameras within IoT platform with 5G networks."
    },
    {
        "title": "Implementation of Autonomous Mobile Robot in SmartFactory",
        "url": "https://mdpi-res.com/d_attachment/applsci/applsci-12-08912/article_deploy/applsci-12-08912-v2.pdf?version=1662454676",
        "abstract": " This study deals with the technology of autonomous mobile robots (AMR) and their implementation on the SmartFactory production line at the Technical University of Ostrava. The task of the mobile robot is to cooperate with the production line, take over the manufactured products, and then deliver them. The content also includes a description of the individual steps that were necessary to make the mobile robot operational, such as loading a virtual map of the space, creating a network for communication with the mobile robot, and programming it. The main part of the experiment deals with testing the accuracy of moving the mobile robot to each position and establishing communication between the production line and the mobile robot. A high accuracy is a necessity in this process. The result of the study is the configuration of the autonomous mobile robot. The repetitive precision of the approach of the autonomous mobile robot to a position is ±3 mm."
    }
]

print("⚡ Pre-calculating collection embeddings locally via SentenceTransformers...")
for doc in PDF_DB:
    # Build a descriptive semantic target string
    combined_metadata = f"Title: {doc['title']}\nAbstract: {doc['abstract']}"
    # Generate the vector locally (returns a numpy array shape (384,))
    doc["embedding"] = local_embedder.encode(combined_metadata, convert_to_numpy=True)
print("✅ Local vector database indexed successfully.")

# Custom UI CSS for a clean, cohesive dark blue environment
dark_blue_css = """
body, .gradio-container { background-color: #0a192f !important; color: #cbd5e1 !important; }
textbox textarea, input { background-color: #172a45 !important; color: #ffffff !important; border: 1px solid #3085fe !important; }
label span { color: #64ffda !important; background-color: transparent !important; }
.prose h1, .prose h2, .prose h3, .prose h4 { color: #ffffff !important; }
"""

def fetch_pdf_bytes(url: str) -> bytes:
    """Streams the raw binary content of the target paper directly into memory."""
    with httpx.Client(follow_redirects=True) as http_client:
        response = http_client.get(url)
        if response.status_code != 200:
            raise RuntimeError(f"Failed to fetch PDF from source. Status code: {response.status_code}")
        return response.content

def format_search_ui(selected_doc, score):
    """Formats the active document panel to display metadata, live abstract, and similarity score."""
    html = f"""
    <div style='font-family: sans-serif;'>
        <h3 style='color: #3085fe; margin-bottom: 10px;'>🎯 Top Matched Grounded Source</h3>
        <div style='border-left: 4px solid #64ffda; background-color: #172a45; padding: 16px; margin-bottom: 15px; border-radius: 0 8px 8px 0; border: 1px solid #1e3a8a;'>
            <a href="{selected_doc['url']}" target="_blank" style="color: #64ffda; font-size: 16px; font-weight: bold; text-decoration: underline; display: block; margin-bottom: 6px;">
                📄 {selected_doc['title']}
            </a>
            <div style="margin-bottom: 10px;">
                <span style="color: #64ffda; font-size: 11px; background-color: rgba(100, 255, 218, 0.1); padding: 2px 6px; border-radius: 4px; margin-right: 8px;">
                    Local Similarity Score: {score:.4f}
                </span>
                <span style="color: #3085fe; font-size: 11px; background-color: rgba(48, 133, 254, 0.1); padding: 2px 6px; border-radius: 4px;">
                    Router: Local all-MiniLM-L6-v2
                </span>
            </div>
            <p style="color: #94a3b8; font-size: 13px; line-height: 1.5; margin: 0; border-top: 1px solid #1e3a8a; padding-top: 8px;">
                <strong>Abstract:</strong> {selected_doc['abstract']}
            </p>
        </div>
    </div>
    """
    return html

def rag_interface(query):
    if not query.strip():
        return "Please input an analysis query.", "Waiting for prompt..."

    try:
        # 1. Generate query vector locally
        query_vector = local_embedder.encode(query, convert_to_numpy=True).reshape(1, -1)

        best_doc = None
        highest_similarity = -1.0

        for doc in PDF_DB:
            doc_vector = doc["embedding"].reshape(1, -1)
            # Efficient matrix similarity math via sklearn
            similarity = cosine_similarity(query_vector, doc_vector)[0][0]

            if similarity > highest_similarity:
                highest_similarity = similarity
                best_doc = doc

        # Update the visual UI verification layout instantly
        search_ui_output = format_search_ui(best_doc, highest_similarity)

        # 2. Fetch binary stream of the selected paper only
        pdf_bytes = fetch_pdf_bytes(best_doc["url"])

        # 3. Pass the file directly to Gemini's native document layer
        pdf_part = types.Part.from_bytes(
            data=pdf_bytes,
            mime_type="application/pdf"
        )

        prompt = f"""
        You are an advanced AI research agent reading an academic paper.
        Analyze the provided full PDF document to answer the user's inquiry thoroughly.
        Ground all assertions in the specific tables, data points, or text found in the file.

        User Query: {query}
        Analysis:
        """

        # 4. Generate grounded synthesis with gemini-2.5-flash
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[pdf_part, prompt]
        )
        ai_response = response.text

    except Exception as e:
        ai_response = f"RAG execution failure: {str(e)}"
        search_ui_output = "<p style='color:#ef4444;'>Failed to complete routing or processing.</p>"

    return search_ui_output, ai_response

# 5. Build Gradio Render Layout
with ui.Blocks(theme=ui.themes.Soft(primary_hue="blue", neutral_hue="slate"), css=dark_blue_css) as demo:
    ui.Markdown(
        """
        # RAG/Search Engine Interface for Research Papers
        ### Live generation powered by processing the full structural PDF documents.
        """
    )

    with ui.Row(equal_height=False):
        query_input = ui.Textbox(
            label="Target Search Question",
            placeholder="e.g., Explain the Mamdani fuzzy logic classification or long-term robot autonomy enablers",
            scale=4
        )
        submit_btn = ui.Button("Ask/Search", variant="primary", scale=1)

    with ui.Row(equal_height=False):
        with ui.Column(scale=1):
            ui.Markdown("### 🧠 AI Analysis Output")
            answer_output = ui.Markdown(value="Enter a query to execute search processing.")
        with ui.Column(scale=1):
            links_output = ui.HTML(value="<p style='color:#64748b;'>Document source verification points will render here.</p>")

    submit_btn.click(
        fn=rag_interface,
        inputs=query_input,
        outputs=[links_output, answer_output]
    )

demo.launch(debug=True)

🧠 Loading local sentence-transformer model ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Pre-calculating collection embeddings locally via SentenceTransformers...
✅ Local vector database indexed successfully.


/tmp/ipykernel_4347/2385507464.py:152: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with ui.Blocks(theme=ui.themes.Soft(primary_hue="blue", neutral_hue="slate"), css=dark_blue_css) as demo:
/tmp/ipykernel_4347/2385507464.py:152: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with ui.Blocks(theme=ui.themes.Soft(primary_hue="blue", neutral_hue="slate"), css=dark_blue_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5e6cc6bb8973a968e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://5e6cc6bb8973a968e6.gradio.live
